## 5. JSON ja SQLite

1. Test Trivia API abil

In [42]:
%pip install requests


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: C:\Users\Lenovo\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [53]:
import requests
import sqlite3
import json
import random
from datetime import datetime

DB_FILE = "trivia_tulemused.db"
JSON_FILE = "trivia_tulemused.json"


In [54]:
def init_db():
    conn = sqlite3.connect(DB_FILE)
    cursor = conn.cursor()
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS results (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        username TEXT,
        category TEXT,
        question_count INTEGER,
        question_type TEXT,
        score INTEGER,
        timestamp TEXT
    )
    """)
    conn.commit()
    conn.close()


Küsimuste kategooria

In [55]:
def get_categories():
    try:
        response = requests.get("https://opentdb.com/api_category.php", timeout=5)
        response.raise_for_status()
        data = response.json()
        if "trivia_categories" in data:
            return data["trivia_categories"]
        else:
            print("Kategooriate andmed puuduvad API vastuses.")
            return []
    except Exception as e:
        print(f"Kategooriate laadimine ebaõnnestus: {e}")
        return []


Küsimuste laadimine

In [56]:
def fetch_questions(amount, category_id, q_type):
    url = f"https://opentdb.com/api.php?amount={amount}&category={category_id}&type={q_type}"
    response = requests.get(url)
    return response.json()["results"]


Küsimuste arv – Kasutaja saab valida, mitu küsimust ta soovib (vahemikus 1–20).
Küsimuste tüüp – Kasutaja saab valida kahe variandi vahel:
multiple – mitme valikuga küsimused
boolean – tõene/vale küsimused

Testi funktsioon / küsimused ja vastused

In [57]:
def run_quiz(questions):
    score = 0
    correct_answers = []

    for i, q in enumerate(questions, 1):
        print(f"{i}. {q['question']}", flush=True)
        if q['type'] == "multiple":
            options = q.get("incorrect_answers", []) + [q.get("correct_answer", "")]
            random.shuffle(options)
        else:
            options = ["True", "False"]

        for idx, option in enumerate(options, 1):
            print(f"  {idx}. {option}", flush=True)

        while True:
            try:
                choice = int(input("Vali vastuse number: "))
                if 1 <= choice <= len(options):
                    break
                else:
                    print("Palun vali korrektne number.")
            except ValueError:
                print("Palun sisesta number.")

        if options[choice - 1] == q["correct_answer"]:
            score += 1

        correct_answers.append((q["question"], q["correct_answer"]))

    return score, correct_answers


Tulemuste salvestamine (SQLite ja JSON)

In [59]:
def save_results(username, category, amount, q_type, score):
    timestamp = datetime.now().isoformat()

    # SQLite
    conn = sqlite3.connect(DB_FILE)
    cursor = conn.cursor()
    cursor.execute("""
        INSERT INTO results (username, category, question_count, question_type, score, timestamp)
        VALUES (?, ?, ?, ?, ?, ?)
    """, (username, category, amount, q_type, score, timestamp))
    conn.commit()
    conn.close()

    # JSON
    try:
        with open(JSON_FILE, "r", encoding="utf-8") as f:
            data = json.load(f)
    except FileNotFoundError:
        data = []

    data.append({
        "username": username,
        "category": category,
        "question_count": amount,
        "question_type": q_type,
        "score": score,
        "timestamp": timestamp
    })

    with open(JSON_FILE, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4)


Statistika

In [60]:
def show_statistics():
    conn = sqlite3.connect(DB_FILE)
    cursor = conn.cursor()
    cursor.execute("SELECT username, score, question_count, timestamp FROM results ORDER BY score DESC LIMIT 5")
    top = cursor.fetchall()

    cursor.execute("SELECT AVG(score * 1.0 / question_count) FROM results")
    avg = cursor.fetchone()[0]
    conn.close()

    print("\nTOP 5 tulemused:")
    for row in top:
        print(f"{row[0]} – {row[1]}/{row[2]} ({row[3]})")

    print(f"\nKeskmine tulemus: {avg:.2%}")


Programm:
Küsimuste kategooria – Laadige saadaolevad kategooriad Trivia API-st.
Küsimuste arv – Kasutaja saab valida, mitu küsimust ta soovib (vahemikus 1–20).
Küsimuste tüüp – Kasutaja saab valida kahe variandi vahel:
multiple – mitme valikuga küsimused
boolean – tõene/vale küsimused
Skoor
Kuupäev ja kellaaeg

In [61]:
init_db()
print("Tere tulemast Trivia testi!")

username = input("Sisesta kasutajanimi: ")
categories = get_categories()

if not categories:
    print("Kategooriate laadimine ebaõnnestus.")
else:
    print("\nKategooriad:")
    for cat in categories:
        print(f"{cat['id']}: {cat['name']}")

    while True:
        try:
            cat_id = int(input("Vali kategooria ID: "))
            category_name = next(cat["name"] for cat in categories if cat["id"] == cat_id)
            break
        except (ValueError, StopIteration):
            print("Palun vali kehtiv kategooria ID.")

    while True:
        try:
            amount = int(input("Mitu küsimust? (1-20): "))
            if 1 <= amount <= 20:
                break
        except ValueError:
            pass
        print("Palun sisesta arv vahemikus 1-20.")

    while True:
        q_type = input("Vali küsimuste tüüp (multiple/boolean): ").strip().lower()
        if q_type in ("multiple", "boolean"):
            break
        print("Palun vali 'multiple' või 'boolean'.")

    questions = fetch_questions(amount, cat_id, q_type)
    if not questions:
        print("Küsimuste laadimine ebaõnnestus.")
    else:
        score, answers = run_quiz(questions)
        print(f"\nSkoor: {score}/{amount}")

        print("\nÕiged vastused:")
        for q, a in answers:
            print(f"- {q} → {a}")

        save_results(username, category_name, amount, q_type, score)
        show_statistics()


Tere tulemast Trivia testi!

Kategooriad:
9: General Knowledge
10: Entertainment: Books
11: Entertainment: Film
12: Entertainment: Music
13: Entertainment: Musicals & Theatres
14: Entertainment: Television
15: Entertainment: Video Games
16: Entertainment: Board Games
17: Science & Nature
18: Science: Computers
19: Science: Mathematics
20: Mythology
21: Sports
22: Geography
23: History
24: Politics
25: Art
26: Celebrities
27: Animals
28: Vehicles
29: Entertainment: Comics
30: Science: Gadgets
31: Entertainment: Japanese Anime & Manga
32: Entertainment: Cartoon & Animations
1. On the periodic table of elements, what is the symbol for Tin?
  1. Na
  2. Ti
  3. Sn
  4. Ni
2. The &#039;Islets of Langerhans&#039; is found in which human organ?
  1. Kidney
  2. Liver
  3. Pancreas
  4. Brain

Skoor: 1/2

Õiged vastused:
- On the periodic table of elements, what is the symbol for Tin? → Sn
- The &#039;Islets of Langerhans&#039; is found in which human organ? → Pancreas

TOP 5 tulemused:
maria 

Küsimuste kategooria – Laadige saadaolevad kategooriad Trivia API-st.

2. Maailma riigid

In [ ]:
%pip install CountryInfo
%pip install pycountry
%pip install questionary

In [ ]:
%pip install countryinfo pycountry pandas InquirerPy


In [62]:
import json
import pandas as pd
from countryinfo import CountryInfo
import pycountry
from pathlib import Path

JSON_FILE = "riigid.json"

Andmete salvestamine (JSON)

In [63]:
def salvesta_andmed():
    riigid = []
    for country in pycountry.countries:
        try:
            info = CountryInfo(country.name)
            data = {
                "name": country.name,
                "population": info.info().get("population", 0),
                "region": info.info().get("region", ""),
                "area": info.info().get("area", 0),
                "currencies": info.info().get("currencies", []),
                "languages": info.info().get("languages", [])
            }
            riigid.append(data)
        except Exception:
            continue
    with open(JSON_FILE, "w", encoding="utf-8") as f:
        json.dump(riigid, f, indent=2, ensure_ascii=False)
    print("Andmed salvestatud faili:", JSON_FILE)

Andmete laadimine

In [64]:
def lae_andmed():
    with open(JSON_FILE, encoding="utf-8") as f:
        data = json.load(f)
    return pd.DataFrame(data)

Riikide kuvamine:

Riigid, mille elanike arv on suurem kui 100 miljonit.
Euroopa riigid.
Riigid, kus ametlik valuuta on EUR.
10 suurimat riiki pindala järgi.
Riigid, millel on rohkem kui 1 ametlik keel.

In [65]:
def kuva_ule_100m(df):
    print(df[df["population"] > 100_000_000][["name", "population"]])

def kuva_euroopa(df):
    print(df[df["region"] == "Europe"][["name", "region"]])

def kuva_eur_valuutaga(df):
    print(df[df["currencies"].apply(lambda c: "EUR" in c)][["name", "currencies"]])

def kuva_suurimad(df):
    print(df.sort_values("area", ascending=False).head(10)[["name", "area"]])

def kuva_mitu_keelt(df):
    print(df[df["languages"].apply(lambda l: len(l) > 1)][["name", "languages"]])

Kasutaja saab valida, milliseid andmeid kuvada. (CLI  - Command Line Interface)

In [66]:
def menu(df):
    while True:
        print("\n Valikud:")
        print("1. Riigid > 100M elanikku")
        print("2. Euroopa riigid")
        print("3. Riigid, mille valuuta on EUR")
        print("4. 10 suurimat riiki pindala järgi")
        print("5. Riigid, millel on rohkem kui 1 ametlik keel")
        print("6. Välju")
        valik = input("Sisesta valik (1-6): ").strip()

        if valik == "1":
            kuva_ule_100m(df)
        elif valik == "2":
            kuva_euroopa(df)
        elif valik == "3":
            kuva_eur_valuutaga(df)
        elif valik == "4":
            kuva_suurimad(df)
        elif valik == "5":
            kuva_mitu_keelt(df)
        elif valik == "6":
            print("Programm lõppenud")
            break
        else:
            print("Tundmatu valik.")
        input("\nVajuta Enter jätkamiseks...")

Faili kontroll
Andmete laadimine
Põhiprogrammi käivitamine

In [67]:
def main():
    if not Path(JSON_FILE).exists():
        print("Laen andmeid, palun oota...")
        salvesta_andmed()
    df = lae_andmed()
    menu(df)

if __name__ == "__main__":
    main()



 Valikud:
1. Riigid > 100M elanikku
2. Euroopa riigid
3. Riigid, mille valuuta on EUR
4. 10 suurimat riiki pindala järgi
5. Riigid, millel on rohkem kui 1 ametlik keel
6. Välju
Tundmatu valik.

 Valikud:
1. Riigid > 100M elanikku
2. Euroopa riigid
3. Riigid, mille valuuta on EUR
4. 10 suurimat riiki pindala järgi
5. Riigid, millel on rohkem kui 1 ametlik keel
6. Välju
Programm lõppenud
